In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
%matplotlib inline

print("Bibliothèques chargées.")

Bibliothèques chargées.


In [2]:
customers = pd.read_csv("../data/processed/customers_clean.csv")
products = pd.read_csv("../data/processed/products_clean.csv")
sales = pd.read_csv("../data/processed/sales_clean.csv")
prediction_churn = pd.read_csv("../data/processed/predictions_churn.csv")
cluster_segment = pd.read_csv("../data/processed/customer_segments_2.csv")
marketing_channel_performance = pd.read_csv("../data/processed/marketing_channel_performance.csv")
categorie_dominante_2 = pd.read_csv("../data/processed/categorie_dominante_2.csv")

In [3]:
prediction_churn.head()

,customer_id,age,frequency,total_spent_history,recency_days,online_ratio,account_tenure_days,churn_probability,churn_prediction,churn_risk_level,marketing_action
0,2001,33,0.0,0.00,303.0,0.000000,1022,0.7222,1,Risque Élevé,Rétention Urgente : Appel VIP / Bon d'achat 20...
1,2002,59,3.0,243.23,20.0,0.333333,572,0.4276,0,Risque Moyen,Réactivation Ciblée : Emailing personnalisé / ...
2,2003,51,5.0,1122.91,39.0,1.000000,1250,0.3694,0,Risque Moyen,Réactivation Ciblée : Emailing personnalisé / ...
3,2004,18,1.0,43.91,181.0,1.000000,849,0.6568,1,Risque Élevé,Rétention Urgente : Appel VIP / Bon d'achat 20...
4,2005,39,7.0,1264.34,18.0,0.857143,636,0.2516,0,Risque Faible,Fidélisation Active : Programme fidélité / Ven...


In [4]:
marketing_channel_performance.head()

,channel,budget,impressions,clicks,conversions,CTR (%),Taux Conversion (%),CPC (€),CPA / CAC (€),CA Estimé (€),ROAS Estimé
0,Email,63248.12,1420580,72450,7120,5.10,9.83,0.87,8.88,605200.0,9.57
1,Online,68940.55,1850320,91200,6850,4.93,7.51,0.76,10.06,582250.0,8.45
2,Social,49120.30,950140,42100,2480,4.43,5.89,1.17,19.81,210800.0,4.29
3,In-Store,41200.75,720450,28500,1650,3.96,5.79,1.45,24.97,140250.0,3.40
4,TV,32500.40,480120,15400,890,3.21,5.78,2.11,36.52,75650.0,2.33


In [5]:
cluster_segment.columns

Index(['customer_id', 'name', 'age', 'gender', 'location', 'join_date',
       'total_spent', 'purchase_frequency', 'total_quantity', 'sales_amount',
       'average_basket', 'segment', 'cluster_2', 'cluster_3'],
      dtype='str')

In [6]:
customer_data = cluster_segment.drop(
    columns=["cluster_2", "cluster_3"]
).copy()

customer_data.head()

,customer_id,name,age,gender,location,join_date,total_spent,purchase_frequency,total_quantity,sales_amount,average_basket,segment
0,2001,Customer_2001,33,Female,New York,2021-12-13,0.00,0.0,0.0,0.00,0.000000,0
1,2002,Customer_2002,59,Male,New York,2023-03-08,1103.79,5.0,9.0,1103.79,220.758000,2
2,2003,Customer_2003,51,Male,Houston,2021-04-29,1172.71,6.0,13.0,1172.71,195.451667,2
3,2004,Customer_2004,18,Male,Houston,2022-06-04,43.91,1.0,1.0,43.91,43.910000,0
4,2005,Customer_2005,39,Female,Los Angeles,2023-01-03,1264.34,7.0,16.0,1264.34,180.620000,2


In [7]:
profil_marketing = (
    customer_data
    .groupby("segment")
    .agg(
        nombre_clients=("customer_id", "count"),
        depense_moyenne=("total_spent", "mean"),
        frequence_moyenne=("purchase_frequency", "mean"),
        panier_moyen=("average_basket", "mean")
    )
    .reset_index()
)

profil_marketing

,segment,nombre_clients,depense_moyenne,frequence_moyenne,panier_moyen
0,0,2962,166.305648,1.742404,74.783075
1,1,1008,1881.721339,11.029762,175.046353
2,2,3030,787.251241,4.528383,199.871597


In [8]:
noms_segments = {
    0: "Clients occasionnels",
    1: "Clients à forte valeur",
    2: "Clients réguliers à panier élevé"
}

profil_marketing["nom_segment"] = (
    profil_marketing["segment"].map(noms_segments)
)

profil_marketing

,segment,nombre_clients,depense_moyenne,frequence_moyenne,panier_moyen,nom_segment
0,0,2962,166.305648,1.742404,74.783075,Clients occasionnels
1,1,1008,1881.721339,11.029762,175.046353,Clients à forte valeur
2,2,3030,787.251241,4.528383,199.871597,Clients réguliers à panier élevé


In [9]:
profil_marketing = profil_marketing.merge(
    categorie_dominante_2,
    on="segment",
    how="left"
)

profil_marketing

,segment,nombre_clients,depense_moyenne,frequence_moyenne,panier_moyen,nom_segment,categorie_dominante,part_categorie
0,0,2962,166.305648,1.742404,74.783075,Clients occasionnels,Lifestyle,29.17
1,1,1008,1881.721339,11.029762,175.046353,Clients à forte valeur,Home,26.58
2,2,3030,787.251241,4.528383,199.871597,Clients réguliers à panier élevé,Home,28.05


In [10]:
strategie_marketing = pd.DataFrame({
    "segment": [0, 1, 2],

    "profil": [
        "Clients occasionnels",
        "Clients à forte valeur",
        "Clients réguliers à panier élevé"
    ],

    "categorie": [
        "Lifestyle",
        "Home",
        "Home"
    ],

    "objectif": [
        "Réactivation",
        "Fidélisation",
        "Développement"
    ],

    "canal": [
        "Email",
        "Email + Online",
        "Email + Online"
    ],

    "contenu": [
        "Promotions et nouveautés Lifestyle",
        "Offres VIP et avantages exclusifs Home",
        "Produits complémentaires et recommandations Home"
    ],

    "action": [
        "Faire revenir les clients",
        "Conserver les clients à forte valeur",
        "Augmenter la valeur des achats"
    ]
})

strategie_marketing

,segment,profil,categorie,objectif,canal,contenu,action
0,0,Clients occasionnels,Lifestyle,Réactivation,Email,Promotions et nouveautés Lifestyle,Faire revenir les clients
1,1,Clients à forte valeur,Home,Fidélisation,Email + Online,Offres VIP et avantages exclusifs Home,Conserver les clients à forte valeur
2,2,Clients réguliers à panier élevé,Home,Développement,Email + Online,Produits complémentaires et recommandations Home,Augmenter la valeur des achats


In [13]:
prediction_churn.head()

,customer_id,age,frequency,total_spent_history,recency_days,online_ratio,account_tenure_days,churn_probability,churn_prediction,churn_risk_level,marketing_action
0,2001,33,0.0,0.00,303.0,0.000000,1022,0.7222,1,Risque Élevé,Rétention Urgente : Appel VIP / Bon d'achat 20...
1,2002,59,3.0,243.23,20.0,0.333333,572,0.4276,0,Risque Moyen,Réactivation Ciblée : Emailing personnalisé / ...
2,2003,51,5.0,1122.91,39.0,1.000000,1250,0.3694,0,Risque Moyen,Réactivation Ciblée : Emailing personnalisé / ...
3,2004,18,1.0,43.91,181.0,1.000000,849,0.6568,1,Risque Élevé,Rétention Urgente : Appel VIP / Bon d'achat 20...
4,2005,39,7.0,1264.34,18.0,0.857143,636,0.2516,0,Risque Faible,Fidélisation Active : Programme fidélité / Ven...


In [16]:
segments_churn = cluster_segment[
    ["customer_id", "segment"]
].merge(
    prediction_churn[
        ["customer_id", "churn_probability", "churn_risk_level"]
    ],
    on="customer_id",
    how="left"
)

segments_churn.head()

,customer_id,segment,churn_probability,churn_risk_level
0,2001,0,0.7222,Risque Élevé
1,2002,2,0.4276,Risque Moyen
2,2003,2,0.3694,Risque Moyen
3,2004,0,0.6568,Risque Élevé
4,2005,2,0.2516,Risque Faible


In [24]:
prediction_churn = prediction_churn.merge(
    cluster_segment[["customer_id", "segment"]],
    on="customer_id",
    how="left"
)

prediction_churn.head()

,customer_id,age,frequency,total_spent_history,recency_days,online_ratio,account_tenure_days,churn_probability,churn_prediction,churn_risk_level,marketing_action,segment
0,2001,33,0.0,0.00,303.0,0.000000,1022,0.7222,1,Risque Élevé,Rétention Urgente : Appel VIP / Bon d'achat 20...,0
1,2002,59,3.0,243.23,20.0,0.333333,572,0.4276,0,Risque Moyen,Réactivation Ciblée : Emailing personnalisé / ...,2
2,2003,51,5.0,1122.91,39.0,1.000000,1250,0.3694,0,Risque Moyen,Réactivation Ciblée : Emailing personnalisé / ...,2
3,2004,18,1.0,43.91,181.0,1.000000,849,0.6568,1,Risque Élevé,Rétention Urgente : Appel VIP / Bon d'achat 20...,0
4,2005,39,7.0,1264.34,18.0,0.857143,636,0.2516,0,Risque Faible,Fidélisation Active : Programme fidélité / Ven...,2


In [25]:
segment_churn = (
    prediction_churn
    .groupby(["segment", "churn_risk_level"])
    .size()
    .reset_index(name="nombre_clients")
)

segment_churn

,segment,churn_risk_level,nombre_clients
0,0,Risque Faible,22
1,0,Risque Moyen,1910
2,0,Risque Élevé,1030
3,1,Risque Faible,899
4,1,Risque Moyen,109
5,2,Risque Faible,544
6,2,Risque Moyen,2365
7,2,Risque Élevé,121


In [26]:
segment_churn["pourcentage"] = (
    segment_churn["nombre_clients"]
    / segment_churn.groupby("segment")["nombre_clients"].transform("sum")
    * 100
).round(2)

segment_churn

,segment,churn_risk_level,nombre_clients,pourcentage
0,0,Risque Faible,22,0.74
1,0,Risque Moyen,1910,64.48
2,0,Risque Élevé,1030,34.77
3,1,Risque Faible,899,89.19
4,1,Risque Moyen,109,10.81
5,2,Risque Faible,544,17.95
6,2,Risque Moyen,2365,78.05
7,2,Risque Élevé,121,3.99


In [27]:
risque_dominant = (
    segment_churn.loc[
        segment_churn.groupby("segment")["pourcentage"].idxmax()
    ]
    [["segment", "churn_risk_level", "pourcentage"]]
    .reset_index(drop=True)
)

risque_dominant


,segment,churn_risk_level,pourcentage
0,0,Risque Moyen,64.48
1,1,Risque Faible,89.19
2,2,Risque Moyen,78.05


In [28]:
def definir_action(risque):
    if risque == "Risque Élevé":
        return "Rétention urgente"
    elif risque == "Risque Moyen":
        return "Réactivation ciblée"
    else:
        return "Fidélisation"

prediction_churn["action_marketing"] = (
    prediction_churn["churn_risk_level"]
    .apply(definir_action)
)

prediction_churn[
    ["customer_id", "segment", "churn_risk_level", "action_marketing"]
].head()

,customer_id,segment,churn_risk_level,action_marketing
0,2001,0,Risque Élevé,Rétention urgente
1,2002,2,Risque Moyen,Réactivation ciblée
2,2003,2,Risque Moyen,Réactivation ciblée
3,2004,0,Risque Élevé,Rétention urgente
4,2005,2,Risque Faible,Fidélisation


In [29]:
cibles_marketing = (
    prediction_churn
    .groupby(["segment", "churn_risk_level", "action_marketing"])
    .size()
    .reset_index(name="nombre_clients")
)

cibles_marketing

,segment,churn_risk_level,action_marketing,nombre_clients
0,0,Risque Faible,Fidélisation,22
1,0,Risque Moyen,Réactivation ciblée,1910
2,0,Risque Élevé,Rétention urgente,1030
3,1,Risque Faible,Fidélisation,899
4,1,Risque Moyen,Réactivation ciblée,109
5,2,Risque Faible,Fidélisation,544
6,2,Risque Moyen,Réactivation ciblée,2365
7,2,Risque Élevé,Rétention urgente,121


In [31]:
cibles_marketing = cibles_marketing.merge(
    categorie_dominante_2[["segment", "categorie_dominante", "part_categorie"]],
    on="segment",
    how="left"
)

cibles_marketing

,segment,churn_risk_level,action_marketing,nombre_clients,categorie_dominante,part_categorie
0,0,Risque Faible,Fidélisation,22,Lifestyle,29.17
1,0,Risque Moyen,Réactivation ciblée,1910,Lifestyle,29.17
2,0,Risque Élevé,Rétention urgente,1030,Lifestyle,29.17
3,1,Risque Faible,Fidélisation,899,Home,26.58
4,1,Risque Moyen,Réactivation ciblée,109,Home,26.58
5,2,Risque Faible,Fidélisation,544,Home,28.05
6,2,Risque Moyen,Réactivation ciblée,2365,Home,28.05
7,2,Risque Élevé,Rétention urgente,121,Home,28.05


In [32]:
noms_segments = {
    0: "Clients occasionnels",
    1: "Clients à forte valeur",
    2: "Clients réguliers à panier élevé"
}

cibles_marketing["nom_segment"] = (
    cibles_marketing["segment"].map(noms_segments)
)

cibles_marketing

,segment,churn_risk_level,action_marketing,nombre_clients,categorie_dominante,part_categorie,nom_segment
0,0,Risque Faible,Fidélisation,22,Lifestyle,29.17,Clients occasionnels
1,0,Risque Moyen,Réactivation ciblée,1910,Lifestyle,29.17,Clients occasionnels
2,0,Risque Élevé,Rétention urgente,1030,Lifestyle,29.17,Clients occasionnels
3,1,Risque Faible,Fidélisation,899,Home,26.58,Clients à forte valeur
4,1,Risque Moyen,Réactivation ciblée,109,Home,26.58,Clients à forte valeur
5,2,Risque Faible,Fidélisation,544,Home,28.05,Clients réguliers à panier élevé
6,2,Risque Moyen,Réactivation ciblée,2365,Home,28.05,Clients réguliers à panier élevé
7,2,Risque Élevé,Rétention urgente,121,Home,28.05,Clients réguliers à panier élevé


In [33]:
def definir_canal(segment):
    if segment == 0:
        return "Email"
    else:
        return "Email + Online"

cibles_marketing["canal_recommande"] = (
    cibles_marketing["segment"].apply(definir_canal)
)

cibles_marketing[
    [
        "nom_segment",
        "churn_risk_level",
        "categorie_dominante",
        "action_marketing",
        "canal_recommande"
    ]
]

,nom_segment,churn_risk_level,categorie_dominante,action_marketing,canal_recommande
0,Clients occasionnels,Risque Faible,Lifestyle,Fidélisation,Email
1,Clients occasionnels,Risque Moyen,Lifestyle,Réactivation ciblée,Email
2,Clients occasionnels,Risque Élevé,Lifestyle,Rétention urgente,Email
3,Clients à forte valeur,Risque Faible,Home,Fidélisation,Email + Online
4,Clients à forte valeur,Risque Moyen,Home,Réactivation ciblée,Email + Online
5,Clients réguliers à panier élevé,Risque Faible,Home,Fidélisation,Email + Online
6,Clients réguliers à panier élevé,Risque Moyen,Home,Réactivation ciblée,Email + Online
7,Clients réguliers à panier élevé,Risque Élevé,Home,Rétention urgente,Email + Online


In [34]:
def definir_contenu(segment, risque):
    if segment == 0:
        return "Promotions et nouveautés Lifestyle"
    
    elif segment == 1:
        return "Offres VIP et nouveautés Home"
    
    else:
        return "Recommandations et produits complémentaires Home"


cibles_marketing["contenu_offre"] = cibles_marketing.apply(
    lambda row: definir_contenu(
        row["segment"],
        row["churn_risk_level"]
    ),
    axis=1
)

cibles_marketing[
    [
        "nom_segment",
        "churn_risk_level",
        "categorie_dominante",
        "action_marketing",
        "canal_recommande",
        "contenu_offre"
    ]
]

,nom_segment,churn_risk_level,categorie_dominante,action_marketing,canal_recommande,contenu_offre
0,Clients occasionnels,Risque Faible,Lifestyle,Fidélisation,Email,Promotions et nouveautés Lifestyle
1,Clients occasionnels,Risque Moyen,Lifestyle,Réactivation ciblée,Email,Promotions et nouveautés Lifestyle
2,Clients occasionnels,Risque Élevé,Lifestyle,Rétention urgente,Email,Promotions et nouveautés Lifestyle
3,Clients à forte valeur,Risque Faible,Home,Fidélisation,Email + Online,Offres VIP et nouveautés Home
4,Clients à forte valeur,Risque Moyen,Home,Réactivation ciblée,Email + Online,Offres VIP et nouveautés Home
5,Clients réguliers à panier élevé,Risque Faible,Home,Fidélisation,Email + Online,Recommandations et produits complémentaires Home
6,Clients réguliers à panier élevé,Risque Moyen,Home,Réactivation ciblée,Email + Online,Recommandations et produits complémentaires Home
7,Clients réguliers à panier élevé,Risque Élevé,Home,Rétention urgente,Email + Online,Recommandations et produits complémentaires Home


In [35]:
strategie_finale = cibles_marketing[
    [
        "segment",
        "nom_segment",
        "churn_risk_level",
        "categorie_dominante",
        "part_categorie",
        "action_marketing",
        "canal_recommande",
        "contenu_offre"
    ]
].copy()

strategie_finale

,segment,nom_segment,churn_risk_level,categorie_dominante,part_categorie,action_marketing,canal_recommande,contenu_offre
0,0,Clients occasionnels,Risque Faible,Lifestyle,29.17,Fidélisation,Email,Promotions et nouveautés Lifestyle
1,0,Clients occasionnels,Risque Moyen,Lifestyle,29.17,Réactivation ciblée,Email,Promotions et nouveautés Lifestyle
2,0,Clients occasionnels,Risque Élevé,Lifestyle,29.17,Rétention urgente,Email,Promotions et nouveautés Lifestyle
3,1,Clients à forte valeur,Risque Faible,Home,26.58,Fidélisation,Email + Online,Offres VIP et nouveautés Home
4,1,Clients à forte valeur,Risque Moyen,Home,26.58,Réactivation ciblée,Email + Online,Offres VIP et nouveautés Home
5,2,Clients réguliers à panier élevé,Risque Faible,Home,28.05,Fidélisation,Email + Online,Recommandations et produits complémentaires Home
6,2,Clients réguliers à panier élevé,Risque Moyen,Home,28.05,Réactivation ciblée,Email + Online,Recommandations et produits complémentaires Home
7,2,Clients réguliers à panier élevé,Risque Élevé,Home,28.05,Rétention urgente,Email + Online,Recommandations et produits complémentaires Home


In [36]:
marketing_channel_performance[
    [
        "channel",
        "budget",
        "CPA / CAC (€)",
        "CA Estimé (€)",
        "ROAS Estimé"
    ]
].sort_values(
    "ROAS Estimé",
    ascending=False
)

,channel,budget,CPA / CAC (€),CA Estimé (€),ROAS Estimé
0,Email,63248.12,8.88,605200.0,9.57
1,Online,68940.55,10.06,582250.0,8.45
2,Social,49120.30,19.81,210800.0,4.29
3,In-Store,41200.75,24.97,140250.0,3.40
4,TV,32500.40,36.52,75650.0,2.33


In [37]:
marketing_channel_performance["poids_roas"] = (
    marketing_channel_performance["ROAS Estimé"]
    / marketing_channel_performance["ROAS Estimé"].sum()
)

marketing_channel_performance[
    ["channel", "ROAS Estimé", "poids_roas"]
]

,channel,ROAS Estimé,poids_roas
0,Email,9.57,0.341298
1,Online,8.45,0.301355
2,Social,4.29,0.152996
3,In-Store,3.40,0.121255
4,TV,2.33,0.083096


In [38]:
budget_total = 100000

marketing_channel_performance["budget_recommande"] = (
    marketing_channel_performance["poids_roas"]
    * budget_total
).round(2)

marketing_channel_performance[
    [
        "channel",
        "ROAS Estimé",
        "poids_roas",
        "budget_recommande"
    ]
].sort_values(
    "budget_recommande",
    ascending=False
)

,channel,ROAS Estimé,poids_roas,budget_recommande
0,Email,9.57,0.341298,34129.81
1,Online,8.45,0.301355,30135.52
2,Social,4.29,0.152996,15299.57
3,In-Store,3.40,0.121255,12125.53
4,TV,2.33,0.083096,8309.56


In [39]:
def definir_objectif(segment, risque):
    if segment == 0:
        if risque == "Risque Élevé":
            return "Réactiver et retenir"
        elif risque == "Risque Moyen":
            return "Réactiver"
        else:
            return "Fidéliser"

    elif segment == 1:
        if risque == "Risque Élevé":
            return "Retenir un client à forte valeur"
        elif risque == "Risque Moyen":
            return "Réactiver un client à forte valeur"
        else:
            return "Fidéliser un client à forte valeur"

    else:
        if risque == "Risque Élevé":
            return "Retenir et réactiver"
        elif risque == "Risque Moyen":
            return "Stimuler les achats"
        else:
            return "Développer le panier"


strategie_finale["objectif_marketing"] = strategie_finale.apply(
    lambda row: definir_objectif(
        row["segment"],
        row["churn_risk_level"]
    ),
    axis=1
)

strategie_finale

,segment,nom_segment,churn_risk_level,categorie_dominante,part_categorie,action_marketing,canal_recommande,contenu_offre,objectif_marketing
0,0,Clients occasionnels,Risque Faible,Lifestyle,29.17,Fidélisation,Email,Promotions et nouveautés Lifestyle,Fidéliser
1,0,Clients occasionnels,Risque Moyen,Lifestyle,29.17,Réactivation ciblée,Email,Promotions et nouveautés Lifestyle,Réactiver
2,0,Clients occasionnels,Risque Élevé,Lifestyle,29.17,Rétention urgente,Email,Promotions et nouveautés Lifestyle,Réactiver et retenir
3,1,Clients à forte valeur,Risque Faible,Home,26.58,Fidélisation,Email + Online,Offres VIP et nouveautés Home,Fidéliser un client à forte valeur
4,1,Clients à forte valeur,Risque Moyen,Home,26.58,Réactivation ciblée,Email + Online,Offres VIP et nouveautés Home,Réactiver un client à forte valeur
5,2,Clients réguliers à panier élevé,Risque Faible,Home,28.05,Fidélisation,Email + Online,Recommandations et produits complémentaires Home,Développer le panier
6,2,Clients réguliers à panier élevé,Risque Moyen,Home,28.05,Réactivation ciblée,Email + Online,Recommandations et produits complémentaires Home,Stimuler les achats
7,2,Clients réguliers à panier élevé,Risque Élevé,Home,28.05,Rétention urgente,Email + Online,Recommandations et produits complémentaires Home,Retenir et réactiver


In [40]:
canvas_marketing = strategie_finale[
    [
        "nom_segment",
        "churn_risk_level",
        "categorie_dominante",
        "objectif_marketing",
        "canal_recommande",
        "contenu_offre"
    ]
].copy()

canvas_marketing

,nom_segment,churn_risk_level,categorie_dominante,objectif_marketing,canal_recommande,contenu_offre
0,Clients occasionnels,Risque Faible,Lifestyle,Fidéliser,Email,Promotions et nouveautés Lifestyle
1,Clients occasionnels,Risque Moyen,Lifestyle,Réactiver,Email,Promotions et nouveautés Lifestyle
2,Clients occasionnels,Risque Élevé,Lifestyle,Réactiver et retenir,Email,Promotions et nouveautés Lifestyle
3,Clients à forte valeur,Risque Faible,Home,Fidéliser un client à forte valeur,Email + Online,Offres VIP et nouveautés Home
4,Clients à forte valeur,Risque Moyen,Home,Réactiver un client à forte valeur,Email + Online,Offres VIP et nouveautés Home
5,Clients réguliers à panier élevé,Risque Faible,Home,Développer le panier,Email + Online,Recommandations et produits complémentaires Home
6,Clients réguliers à panier élevé,Risque Moyen,Home,Stimuler les achats,Email + Online,Recommandations et produits complémentaires Home
7,Clients réguliers à panier élevé,Risque Élevé,Home,Retenir et réactiver,Email + Online,Recommandations et produits complémentaires Home


In [41]:
# Sauvegarder le résultat de la segmentation

canvas_marketing = canvas_marketing.copy()

canvas_marketing.to_csv(
    "../data/processed/canvas_marketing.csv",
    index=False
)